# Baseline CNN

A from-scratch PyTorch CNN trained on log-mel spectrograms,
using the shared `carol_anne` preprocessing pipeline and the fixed
`data/manifest.csv` train/val/test split. This is a first-pass baseline meant
to be iterated on, not a final model.

Spectrograms are saved into a cache directory inside data/ so that kernel restarts do not need to rebuild them.


## Imports

Standard imports plus the shared `carol_anne` labels and preprocessing helpers. Sets a fixed random seed and picks the best available device (cuda, mps, or cpu).

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Audio

import librosa

from carol_anne.labels import IDX_TO_LABEL, LABEL_TO_IDX, LABELS
from carol_anne.preprocess import load_clip, logmel, SR, SECONDS, N_MELS

torch.manual_seed(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")


## Data Importing & Preprocessing

Loads the manifest and splits it into train, val, and test using the split already assigned in that file. Builds the label to index lookup used everywhere else in the notebook.

In [ ]:
DATA_DIR = "<PLACEHOLDER: path to the folder of data files>"

manifest = pd.read_csv("<PLACEHOLDER: path to manifest.csv>")

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

## Dataset Testing | Natural-Length Evaluation

Val and test have always gone through some fixed-window fallback (loop-fill or zero-pad) whenever a clip is shorter than the window, which is most clips. A real user will never upload a looped or silence-padded recording, so evaluating on that data isn't fully honest about deployed performance, even if it comes close.

Training keeps its existing recipe unchanged. PlaceholderDatasetChunked below (4s chunks, loop-filled short clips and remainders) is reused as is, since synthetically expanding or manipulating training data is standard practice and doesn't raise the same "will a user ever see this" concern.

Val and test change fundamentally. PlaceholderDatasetNatural returns each clip's own full, unmodified spectrogram: no fixed window, no crop, no pad, no loop, nothing synthetic. This works because CNNBaseline's last feature layer is nn.AdaptiveAvgPool2d((1, 1)) (see the CNN Architecture section below), which pools any spatial size down to a fixed 1x1 before the classifier ever sees it. The model was never architecturally constrained to a fixed window; that constraint only ever existed for batch collation. Since these tensors are now variable-shaped, val_loader and test_loader switch to batch_size=1 in the DataLoaders section below (PyTorch's default collate can't stack differently-shaped tensors). This costs nothing here, since val and test are only 448 and 447 samples, and model.eval() already uses BatchNorm's running stats rather than per-batch stats.

It uses its own cache directory (../data/cache_natural) for val and test. Training keeps reading from the existing ../data/cache_chunked cache from the loop-fill chunking run, so none of that caching work gets repeated.

In [ ]:
CHUNK_SECONDS = 4.0


def _loop_fill(y, window, crossfade_ms=25, sr=SR):
    """Fills `window` by forward-looping y, crossfading each seam so the splice
    is inaudible instead of leaving a hard discontinuity (or, worse, silence).
    Same helper as the earlier chunking section, redefined here so this
    section stays self-contained now that section is commented out."""
    crossfade_len = min(int(sr * crossfade_ms / 1000), len(y) // 2)

    if crossfade_len <= 0:
        # too short to crossfade meaningfully - fall back to plain tiling
        n_repeats = int(np.ceil(window / len(y)))
        return np.tile(y, n_repeats)[:window]

    fade_out = np.linspace(1.0, 0.0, crossfade_len)
    fade_in = np.linspace(0.0, 1.0, crossfade_len)

    out = y.copy()
    while len(out) < window:
        # blend the tail of what we've built so far with the head of the next
        # forward repetition of y, so the loop point has no audible click
        tail = out[-crossfade_len:] * fade_out
        head = y[:crossfade_len] * fade_in
        out = np.concatenate([out[:-crossfade_len], tail + head, y[crossfade_len:]])

    return out[:window]


def chunk_clip(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
    """Splits a clip into consecutive, non-overlapping chunk_seconds windows,
    covering the whole clip in order instead of centering one big window on
    the loudest moment. The trailing remainder (or the whole clip, if it's
    shorter than one chunk) is filled up to a full window with _loop_fill, so
    every returned chunk is exactly chunk_seconds long, but only the fill
    portion (if any) of any given chunk is looped content. Training only -
    val/test use PlaceholderDatasetNatural below, with no windowing
    or fill of any kind."""
    y, sr = librosa.load(path, sr=sr, mono=True)
    sr = int(sr)
    window = int(chunk_seconds * sr)

    n_full_chunks = len(y) // window
    chunks = [y[i * window:(i + 1) * window] for i in range(n_full_chunks)]

    remainder = y[n_full_chunks * window:]
    if len(remainder) > 0:
        chunks.append(_loop_fill(remainder, window, sr=sr))

    return chunks


class PlaceholderDatasetChunked(Dataset):
    """Training dataset (unchanged from the loop-fill chunking section above):
    each clip contributes one example per chunk_clip() window instead of
    exactly one example per clip - a clip longer than chunk_seconds yields
    several real (non-looped) chunks, while a clip shorter than chunk_seconds
    still yields exactly one, loop-filled.

    Building the index up front (one librosa.get_duration call per row, which
    reads duration without decoding the full clip) is what lets __len__ - and
    therefore the preload() progress bar - report the true chunk count before
    any audio is actually loaded."""

    def __init__(self, df, data_dir=DATA_DIR, cache_dir="<PLACEHOLDER: spectrogram cache folder (cache_chunked)>", chunk_seconds=CHUNK_SECONDS):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.cache_dir = cache_dir
        self.chunk_seconds = chunk_seconds
        self._cache = {}

        # one (row_idx, chunk_idx) entry per training example this dataset yields;
        # built from this df alone, so chunks never cross the train/val/test split
        # already encoded in which df (train_df/val_df/test_df) was passed in
        self._index = []
        for row_idx, row in self.df.iterrows():
            duration = librosa.get_duration(path=os.path.join(self.data_dir, row["path"]))
            n_chunks = max(1, int(np.ceil(duration / chunk_seconds)))
            self._index.extend((row_idx, chunk_idx) for chunk_idx in range(n_chunks))

    def __len__(self):
        return len(self._index)

    def _chunk_cache_path(self, row_idx, chunk_idx):
        row = self.df.iloc[row_idx]
        rel_path = os.path.splitext(row["path"])[0] + f"_chunk{chunk_idx}.npy"
        return os.path.join(self.cache_dir, rel_path)

    def _compute_and_cache_row_chunks(self, row_idx):
        """Decodes a clip once and caches every one of its chunks in a single
        pass, so requesting chunk 1 right after chunk 0 doesn't re-decode the
        same file - decoding audio is the expensive part, not slicing it."""
        row = self.df.iloc[row_idx]
        path = os.path.join(self.data_dir, row["path"])
        chunks = chunk_clip(path, chunk_seconds=self.chunk_seconds)

        mels = []
        for chunk_idx, chunk in enumerate(chunks):
            mel = logmel(chunk)
            cache_path = self._chunk_cache_path(row_idx, chunk_idx)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            np.save(cache_path, mel)
            self._cache[(row_idx, chunk_idx)] = mel
            mels.append(mel)
        return mels

    def _load_mel(self, row_idx, chunk_idx):
        key = (row_idx, chunk_idx)
        if key in self._cache:
            return self._cache[key]

        cache_path = self._chunk_cache_path(row_idx, chunk_idx)
        if os.path.exists(cache_path):
            mel = np.load(cache_path)
            self._cache[key] = mel
            return mel

        mels = self._compute_and_cache_row_chunks(row_idx)
        # librosa.get_duration (used to size self._index in __init__) reads
        # duration from the mp3's header and can very slightly overestimate it
        # for some VBR-encoded files, occasionally predicting one more chunk
        # than chunk_clip() actually produces once the audio is really decoded.
        # Clamp instead of crashing: the "missing" chunk is a rounding
        # artifact, not real audio content being lost.
        return mels[min(chunk_idx, len(mels) - 1)]

    def __getitem__(self, idx):
        row_idx, chunk_idx = self._index[idx]
        mel = self._load_mel(row_idx, chunk_idx)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
        label = LABEL_TO_IDX[self.df.iloc[row_idx]["label"]]
        return x, label

    def preload(self, progress_widget=None):
        for i in range(len(self)):
            row_idx, chunk_idx = self._index[i]
            self._load_mel(row_idx, chunk_idx)
            if progress_widget is not None:
                progress_widget.value = i + 1


class PlaceholderDatasetNatural(Dataset):
    """Val/test dataset: each clip's own full, natural-length spectrogram -
    no crop, no pad, no loop, no synthetic content of any kind. This is
    exactly what a deployed model receives from a real user upload, so
    metrics measured this way have 100% data integrity.

    __getitem__ returns variable-shaped tensors (frames scales with each
    clip's own duration) since there's no fixed window at all - a DataLoader
    built from this class MUST use batch_size=1 (see the DataLoaders section
    below), since the default collate can't stack differently-shaped tensors
    into one batch. That's fine here: only 448/447 samples, and CNNBaseline's
    AdaptiveAvgPool2d already makes single-sample forward passes work
    regardless of input width."""

    def __init__(self, df, data_dir=DATA_DIR, cache_dir="<PLACEHOLDER: spectrogram cache folder (cache_natural)>"):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.cache_dir = cache_dir
        self._cache = {}

    def __len__(self):
        return len(self.df)

    def _cache_path(self, idx):
        row = self.df.iloc[idx]
        rel_path = os.path.splitext(row["path"])[0] + ".npy"
        return os.path.join(self.cache_dir, rel_path)

    def _load_mel(self, idx):
        if idx in self._cache:
            return self._cache[idx]

        cache_path = self._cache_path(idx)
        if os.path.exists(cache_path):
            mel = np.load(cache_path)
        else:
            row = self.df.iloc[idx]
            path = os.path.join(self.data_dir, row["path"])
            y, _ = librosa.load(path, sr=SR, mono=True)  # no window, no crop, no fill of any kind
            mel = logmel(y)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            np.save(cache_path, mel)

        self._cache[idx] = mel
        return mel

    def __getitem__(self, idx):
        mel = self._load_mel(idx)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames) - frames varies per clip
        label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
        return x, label

    def preload(self, progress_widget=None):
        for i in range(len(self)):
            self._load_mel(i)
            if progress_widget is not None:
                progress_widget.value = i + 1

## Train, Test, Validate Datasets

Builds the three PyTorch datasets and preloads every spectrogram into the cache, with a progress bar per split. Slow the first time, fast on later runs once the cache is there.

In [ ]:
train_ds = PlaceholderDatasetChunked(train_df)
val_ds = PlaceholderDatasetNatural(val_df)
test_ds = PlaceholderDatasetNatural(test_df)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    progress = widgets.IntProgress(min=0, max=len(ds), description=f"{name}:")
    display(progress)
    ds.preload(progress)

## Sanity Check Spectrogram

Pulls one training sample and plots its spectrogram, just to check the shape and values look right before training starts.

In [ ]:
x, y = train_ds[0]
print(f"Sample tensor shape: {x.shape}, dtype: {x.dtype}, label: {IDX_TO_LABEL[y]}")

plt.figure(figsize=(6, 3))
plt.imshow(x[0].numpy(), origin="lower", aspect="auto")
plt.title(f"Example spectrogram — {IDX_TO_LABEL[y]}")
plt.xlabel("time frame")
plt.ylabel("mel bin")
plt.colorbar()
plt.show()

## DataLoaders

Groupg individual examples into batches for training. Only the training loader shuffles: val/test order doesn't matter and we want it stable so we can line predictions back up with val_df/test_df rows later.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
# val/test now yield each clip's own natural, variable-length spectrogram (no
# fixed window at all) - the default collate can't stack differently-shaped
# tensors into a batch, so these use batch_size=1. No real cost here: only
# 448/447 samples, and model.eval() already uses BatchNorm's running stats
# rather than per-batch stats, so batch_size=1 isn't a training-time concern.
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)


## CNN Architecture

Imports `CNNBaseline` from `src/carol_anne/models/cnn.py` instead of defining it here, so the notebook and the deployed API use the exact same model code.

In [ ]:
from carol_anne.models.cnn import CNNBaseline

## Instantiate Model

Creates the model, moves it to the chosen device, and prints the total parameter count.

In [ ]:
model = CNNBaseline().to(device)
n_params = sum(p.numel() for p in model.parameters())

print(f"Model has {n_params:,} parameters")

## Loss, Optimizer & Hyperparameters

Sets the epoch count and learning rate, then defines the loss function, optimizer, and the scheduler that halves the learning rate when val macro F1 stalls. Also sets where the best checkpoint gets saved.

In [ ]:
N_EPOCHS = 20
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# Tracks val_macro_f1 (the same metric already used for checkpoint selection below) and
# halves LR whenever it stalls for `patience` epochs, to stabilize the late-training
# oscillation seen with a constant 1e-3 LR.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-5
)

os.makedirs("<PLACEHOLDER: folder for the model checkpoint>", exist_ok=True)
CHECKPOINT_PATH = "<PLACEHOLDER: path to cnn_baseline_best.pt>"

## Training

Runs the full training loop: one epoch of training, then a pass over val, logging the metrics and saving a checkpoint whenever val macro F1 improves. Prints one line per epoch.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_macro_f1": [], "lr": []}
best_val_f1 = -1.0

for epoch in range(N_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == y).sum().item()
            val_total += x.size(0)
            y_true.extend(y.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_macro_f1 = f1_score(y_true, y_pred, average="macro")

    # steps the LR down when val_macro_f1 stalls; must run after val_macro_f1 is known
    # and before checkpointing so a just-lowered LR is reflected in this epoch's log line
    scheduler.step(val_macro_f1)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(f"epoch {epoch+1}/{N_EPOCHS}  train_loss={train_loss:.3f}  "
          f"val_loss={val_loss:.3f}  val_acc={val_acc:.3f}  val_macro_f1={val_macro_f1:.3f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(
            {"state_dict": model.state_dict(), "labels": LABELS,
             "val_macro_f1": val_macro_f1, "epoch": epoch},
            CHECKPOINT_PATH,
        )


print(f"Best val macro F1: {best_val_f1:.4f}")

## Testing / Verification

Loads the best saved checkpoint and runs it once over the test set, collecting true labels, predicted labels, and predicted probabilities for the Metrics section below.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(1)
        y_true.extend(y.tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.extend(probs.cpu().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)


## Metrics

Prints test accuracy, macro F1, weighted F1, and the three weakest classes, then shows a color coded table with precision, recall, and F1 for every class.

In [ ]:
from typing import cast

# cast: classification_report's stub doesn't narrow to dict from a plain bool
# output_dict=True, so Pylance infers the str-returning overload instead and
# flags .pop()/string indexing below as invalid - a stub limitation, not a
# runtime issue (output_dict=True always returns a dict at runtime).
report = cast(dict, classification_report(y_true, y_pred, target_names=LABELS, output_dict=True, zero_division=0))
accuracy = report.pop("accuracy")
macro_f1 = report["macro avg"]["f1-score"]
weighted_f1 = report["weighted avg"]["f1-score"]

per_class_df = pd.DataFrame(report).T.drop(index=["macro avg", "weighted avg"]).sort_values("f1-score")
weakest = per_class_df.head(3)

print(f"Test accuracy:    {accuracy:.4f}")
print(f"Test macro F1:    {macro_f1:.4f}  <- headline metric (unweighted mean across classes)")
print(f"Test weighted F1: {weighted_f1:.4f}  (weighted by class support; close to macro since classes are near-balanced)")
print("Weakest classes:  " + ", ".join(f"{cls} ({row['f1-score']:.2f})" for cls, row in weakest.iterrows()))

# sorted weakest-first so the classes needing attention are the first thing you see;
# background_gradient is static HTML (no ipywidgets), so it still renders in a plain
# GitHub/nbviewer preview, unlike the interactive dashboard below
display(
    per_class_df.style
    .background_gradient(subset=["precision", "recall", "f1-score"], cmap="RdYlGn", vmin=0, vmax=1)
    .format({"precision": "{:.2f}", "recall": "{:.2f}", "f1-score": "{:.2f}", "support": "{:.0f}"})
    .set_caption("Per-class metrics, sorted weakest first")
)

### Baseline Model Metrics (historical reference)



Log a row here each time a baseline/fine-tuned variant finishes a run, using the values printed in [Metrics](#Metrics) above (`accuracy`, `macro_f1`, `weighted_f1`, `report["macro avg"]`, `weakest`) plus `N_EPOCHS` and `n_params`. Keeps a fixed reference point to compare new runs against as configs change.

| Date | Model / Config | Epochs | Params | Test Accuracy | Macro F1 | Weighted F1 | Macro Precision | Macro Recall | Weakest Classes (F1) | Notes |
|---|---|---|---|---|---|---|---|---|---|---|
| _YYYY-MM-DD_ | _e.g. baseline v1 (dropout=0.3, lr=1e-3)_ | _e.g. 20_ | _e.g. 106,538_ | _0.XX_ | _0.XX_ | _0.XX_ | _0.XX_ | _0.XX_ | _Class01 (0.XX), Class02 (0.XX)_ | _notes_ |
| 2026-09-10 | baseline v1 (dropout=0.3, lr=1e-3) | 20 | 106,538 | 0.6890 | 0.6954 | 0.6954 | ~0.74 | ~0.69 | Class04 (0.53), Class08 (0.53), Class06 (0.56) | from last executed run (cell `341c77b1`); macro precision/recall derived from the displayed 2-decimal per-class values, so approximate |
| 2026-09-10 | forward-loop dataset (crossfade fill, dropout=0.3, lr=1e-3) | 20 | 106,538 | 0.8210 | 0.8192 | 0.8194 | 0.827 | 0.820 | Class08 (0.62), Class04 (0.73), Class10 (0.78) | `PlaceholderDatasetLooped` replacing zero-padding with a forward loop + 25ms crossfade; accumulated readings from this run vs. baseline v1 above: accuracy +0.132, macro F1 +0.124 |
| 2026-09-10 | LR scheduler (ReduceLROnPlateau) + SpecAugment, freq=10/time=20 (dropout=0.3, lr=1e-3→5e-4→2.5e-4) | 20 | 106,538 | 0.8076 | 0.8052 | 0.8052 | 0.810 | 0.807 | Class08 (0.57), Class04 (0.66), Class06 (0.73) | val_macro_f1 in the last 5 epochs settled to 0.79–0.81 (vs. wild 0.665–0.829 swings in the prior run) — scheduler fixed the late-training instability, but the peak/test score is slightly below the forward-loop run above, likely because SpecAugment makes each epoch harder and 20 epochs wasn't enough to fully recover; worth trying more epochs or lighter masking (see notebook discussion) |
| 2026-09-10 | LR scheduler (ReduceLROnPlateau) + SpecAugment, freq=5/time=10 (dropout=0.3, lr=1e-3→5e-4→2.5e-4) | 20 | 106,538 | 0.8188 | 0.8175 | 0.8175 | 0.819 | 0.819 | Class08 (0.65), Class04 (0.67), Class10 (0.76) | halved the mask sizes from the row above; recovered almost all of the regression and now roughly matches the forward-loop run (accuracy -0.002, macro F1 -0.002) while keeping the scheduler's stabilized late-training curve (best val macro F1 0.8336, the best of any run so far) — lighter SpecAugment + LR scheduling together look like a small net win over forward-loop alone |
| 2026-09-10 | Audio chunking, CHUNK_SECONDS=4.0, loop-fill, **windowed val/test** (train: `PlaceholderDatasetChunked`, val/test: `PlaceholderDatasetSingleChunk`; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.8389 | 0.8391 | 0.8391 | 0.843 | 0.839 | Class08 (0.71), Class04 (0.76), Class01 (0.77) | Best-scoring run so far, but val/test still used a synthetic loudest-centered 4s window (loop-filled when short) — not what a real user's upload looks like. Superseded for reporting purposes by the natural-length eval run below; kept as the record of the ceiling this windowing approach could reach |
| 2026-09-10 | Audio chunking, CHUNK_SECONDS=4.0, **no loop** / zero-pad fill, windowed val/test (train: `PlaceholderDatasetChunkedNoLoop`, val/test: `PlaceholderDatasetSingleChunkNoLoop`; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.7830 | 0.7834 | 0.7834 | 0.798 | 0.783 | Class08 (0.60), Class06 (0.68), Class04 (0.69) | Same 2996-chunk train set/window/schedule as the loop-fill run above, only the fill strategy changed (silence instead of crossfaded loop). Clearly worse across the board: -0.056 accuracy, -0.056 macro F1 vs. the loop-fill run, and train_loss stayed markedly higher all 20 epochs (0.740 final vs. 0.513 for loop-fill) - the model visibly struggled more to fit silence-padded inputs. Answers the "does looping still matter at 4s?" question: yes, clearly - looping is not just a vestige of the old 11s window, it's actively still worth ~5-6 points at 4s too |
| 2026-09-10 | Audio chunking train (loop-fill, unchanged) + **natural-length, no-window val/test** (train: `PlaceholderDatasetChunked`, val/test: `PlaceholderDatasetNatural`, batch_size=1; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | **0.8210** | **0.8211** | 0.8211 | 0.832 | 0.821 | Class08 (0.69), Class04 (0.73), Class01 (0.75) | **First 100%-honest evaluation**: val/test use each clip's own real, unmodified audio - no crop, no pad, no loop, nothing synthetic (`nn.AdaptiveAvgPool2d((1,1))` means the CNN never actually required a fixed window; that constraint only existed for batch collation, so val/test now run at batch_size=1 instead). 1.8 points below the windowed-eval run above, which makes sense: that run benefited from evaluating on the same loudest-centered, length-normalized distribution the model trained on. This 0.8211 is the trustworthy number going forward - very close to the original forward-loop run's 0.8192, meaning most of the apparent gains from chunking/window-size tuning were partly an artifact of val/test also getting easier, not just the model getting better. Should be the baseline for judging any future change from here on |
| 2026-09-14 | Audio chunking train (loop-fill, unchanged) + natural-length, no-window val/test, re-run for a fresh baseline (train: `PlaceholderDatasetChunked`, val/test: `PlaceholderDatasetNatural`, batch_size=1; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.8322 | 0.8328 | 0.8327 | ~0.85 | ~0.83 | Class08 (0.73), Class04 (0.74), Class10 (0.76) | Same config as the row above; best val macro F1 0.8318. Macro precision/recall derived from the displayed 2-decimal per-class values, so approximate |
| 2026-09-14 | Same natural-length pipeline, `CNNBaseline` now imported from `src/carol_anne/models/cnn.py` instead of defined inline (train: `PlaceholderDatasetChunked`, val/test: `PlaceholderDatasetNatural`, batch_size=1; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.8166 | 0.8153 | 0.8154 | ~0.82 | ~0.82 | Class08 (0.63), Class04 (0.67), Class10 (0.80) | Confirms the `cnn.py` import path produces the same architecture and training behavior as the inline class; best val macro F1 0.8480. One rough epoch (epoch 2: val_loss spiked to 4.080, val_acc dropped to 0.143) before recovering, normal run-to-run variance, not related to the import change. Macro precision/recall derived from the displayed 2-decimal per-class values, so approximate |

Class names in the table follow the order in `labels.py`: `Class01` is the first label, `Class02` the second, and so on.

## Analytics Dashboard

Builds the interactive dashboard: training curves, confusion matrix, per class breakdown, duration bias check, UMAP and PCA of the learned features, and Grad-CAM saliency maps, all together in one grid.

In [ ]:
from typing import cast

from carol_anne.cnn_analytics import (
    build_training_curves_panel, build_confusion_matrix_panel, build_per_class_panel,
    build_sample_browser_panel, build_duration_bias_panel, build_umap_panel, build_pca_panel,
    GradCAM, build_saliency_panel, build_analytics_grid, close_widget_tree,
)

def extract_features(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            feats.append(model.features(x).flatten(1).cpu().numpy())
            labels.extend(y.tolist())
    return np.concatenate(feats), np.array(labels)

test_features, test_labels = extract_features(model, test_loader)
durations = [librosa.get_duration(path=os.path.join(DATA_DIR, p)) for p in test_df["path"]]

# tear down the previous run's hooks before registering a new pair on the same
# layer, so re-running this cell doesn't stack hooks on the model forever
old_grad_cam = globals().get("grad_cam")
if old_grad_cam is not None:
    old_grad_cam.remove_hooks()

# required: full_backward_hook conflicts with in-place ReLU. Cast to nn.ReLU since
# nn.Sequential.__getitem__ is typed as returning the generic nn.Module, which
# doesn't statically expose `.inplace` (Pylance false positive, not a runtime issue).
cast(nn.ReLU, model.features[-2]).inplace = False
grad_cam = GradCAM(model, model.features[-2])

def to_input_tensor(mel):
    return torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).float().to(device)

# only set on the very first run - a real grid built by the cell below must survive
# subsequent re-runs of *this* cell so that cell can still close it before rebuilding
if "grid" not in globals():
    grid = None

In [ ]:
close_widget_tree(grid)  # tear down the previous run's grid before displaying a fresh one, so it doesn't linger and double up

grid = build_analytics_grid([
    (0, 0, build_training_curves_panel(history)),
    (0, 1, build_confusion_matrix_panel(y_true, y_pred, LABELS)),
    (1, 0, build_per_class_panel(y_true, y_pred, LABELS)),
    (1, 1, build_duration_bias_panel(y_true, y_pred, durations, LABELS)),
    (2, 0, build_umap_panel(test_features, y_true, y_pred, LABELS)),
    (2, 1, build_pca_panel(test_features, y_true, y_pred, LABELS)),
    (3, 0, build_saliency_panel(
        grad_cam, LABELS, y_true, y_pred,
        get_normalized_mel=lambda i: (lambda m: (m - m.mean()) / (m.std() + 1e-6))(test_ds._load_mel(i)),
        to_input_tensor=to_input_tensor, num_samples=len(test_df))),
])
display(grid)

## Sample Inspector

Pulled out of the grid above into its own full-width section. It's the main tool
for browsing individual test clips (by class or misclassifications only), so it
needs more room than a single grid cell for its waveform, spectrogram, probability
bars, and audio player to all be legible at once.

In [ ]:
sample_browser = build_sample_browser_panel(
    LABELS, y_true, y_pred, y_prob,
    # the clip's real, unmodified audio - matches what test_ds actually evaluated on,
    # not the old fixed-11s-window load_clip() which no longer reflects the eval pipeline
    get_waveform=lambda i: librosa.load(os.path.join(DATA_DIR, test_df.iloc[i]["path"]), sr=SR, mono=True)[0],
    get_mel=lambda i: test_ds._load_mel(i),
    sample_rate=SR, num_samples=len(test_df),
)
sample_browser.layout.width = "100%"
sample_browser.layout.border = "1px solid rgba(128, 128, 128, 0.4)"
sample_browser.layout.padding = "8px"
display(sample_browser)